# Assess Morphology Signature Significance

This notebook measures the statistical significance and effect size of each morphology feature in the CFReT dataset. We compare healthy and failing cardiomyocyte cells (both treated with DMSO) to identify which features differ between the two conditions.

For each morphology feature, we run a two-sample Kolmogorov-Smirnov (KS) test and apply Benjamini-Hochberg FDR correction. Features that pass the significance threshold are labeled **"on"** (part of the morphology signature), while the rest are labeled **"off"**.

We also run the same analysis on shuffled data as a null control to confirm that the features identified in the observed data are not due to chance.

In [1]:
import sys
import pathlib

import polars as pl

sys.path.append("../../")
from utils.io_utils import load_profiles, load_configs
from scipy.stats import ks_2samp
from statsmodels.stats.multitest import fdrcorrection
from utils.data_utils import split_meta_and_features
from utils.data_utils import shuffle_feature_profiles

## KS Test Function

`compute_ks_signature` runs a two-sample KS test for each morphology feature between a reference and a target population. It applies FDR correction to control for multiple testing, then labels each feature as **"on"** (statistically significant) or **"off"** (not significant). Results are saved to a CSV file.

In [2]:
def compute_ks_signature(
    ref_df: pl.DataFrame,
    target_df: pl.DataFrame,
    features: list[str],
    output_path: pathlib.Path,
    alpha: float = 0.05,
) -> pl.DataFrame:
    """Run KS test on each feature between ref and target, apply FDR correction,
    and save results to a CSV.

    Parameters
    ----------
    ref_df : pl.DataFrame
        Reference population DataFrame.
    target_df : pl.DataFrame
        Target population DataFrame.
    features : list[str]
        Feature column names to test.
    output_path : pathlib.Path
        Path to write the resulting CSV.
    alpha : float
        Significance threshold for the "on"/"off" signature label.

    Returns
    -------
    pl.DataFrame
        KS test results with FDR-corrected p-values, -log10 transform, signature
        label, and channel.
    """
    # known channels in the dataset
    KNOWN_CHANNELS = {"Actin", "ER", "Hoechst", "Mitochondria", "PM"}
    channel_pattern = "|".join(KNOWN_CHANNELS)
    ks_stats, p_values = zip(
        *[ks_2samp(ref_df[feat], target_df[feat]) for feat in features]
    )

    _, p_values_fdr = fdrcorrection(list(p_values))

    results_df = (
        pl.DataFrame(
            {
                "feature": features,
                "p_value": list(p_values),
                "ks_stat": list(ks_stats),
                "p_value_fdr_corrected": p_values_fdr,
            }
        )
        .with_columns(
            (-pl.col("p_value_fdr_corrected").log10()).alias("neg_log10_p_value")
        )
        .with_columns(
            pl.when(pl.col("p_value_fdr_corrected") < alpha)
            .then(pl.lit("on"))
            .otherwise(pl.lit("off"))
            .alias("signature")
        )
        .with_columns(pl.col("feature").str.split("_").list.get(0).alias("compartment"))
    )

    # add channel extraction logic here if needed, e.g. using regex to extract known channels
    results_df = results_df.with_columns(
        [
            pl.col("feature")
            .str.extract(rf"_({channel_pattern})(?:_|$)", group_index=1)
            .fill_null("no-channel")
            .alias("channel")
        ]
    )

    results_df.write_csv(output_path)
    return results_df

## Setup

Define file paths for the input data and create output directories for saving the signature results.

In [3]:
# load raw data paths
cfret_data_dir = pathlib.Path("../0.download-data/data/sc-profiles/cfret/").resolve(
    strict=True
)
cfret_profiles_path = (
    cfret_data_dir / "localhost230405150001_sc_feature_selected.parquet"
).resolve(strict=True)
cfret_feature_space_path = (
    cfret_data_dir / "cfret_feature_space_configs.json"
).resolve(strict=True)

# create results directory
results_dir = pathlib.Path("./results").resolve()
results_dir.mkdir(parents=True, exist_ok=True)

# create subdirectory for signature outputs
signatures_results_dir = pathlib.Path(results_dir / "signatures")
signatures_results_dir.mkdir(exist_ok=True)

## Parameters

Set the column names and labels used to identify the healthy and failing cell populations, and the method used to compute the on/off signature labels.

In [4]:
# column that holds the combined cell type and treatment label
treatment_col = "Metadata_cell_type_and_treatment"

# population labels used to split cells into reference and target groups
healthy_label = "healthy_DMSO"  # target: healthy cardiomyocytes treated with DMSO
failing_label = "failing_DMSO"  # reference: failing cardiomyocytes treated with DMSO

# statistical method used to compute signature feature importance
on_off_signatures_method = "ks_test"

## Load Data

Load the CFReT single-cell profiles and restrict columns to the relevant metadata and morphology features. A combined cell-type-and-treatment label is added for easy population filtering.

In [5]:
# loading profiles
cfret_df = load_profiles(cfret_profiles_path)

# load cfret_df feature space and update cfret_df
cfret_feature_space = load_configs(cfret_feature_space_path)
cfret_meta_features = cfret_feature_space["metadata-features"]
cfret_features = cfret_feature_space["morphology-features"]
cfret_df = cfret_df.select(pl.col(cfret_meta_features + cfret_features))

# add another metadata column that combins both Metadata_heart_number and Metadata_treatment
cfret_df = cfret_df.with_columns(
    (
        pl.col("Metadata_treatment").cast(pl.Utf8)
        + "_heart_"
        + pl.col("Metadata_heart_number").cast(pl.Utf8)
    ).alias("Metadata_treatment_and_heart")
)

# renaming Metadata_treatment to Metadata_cell_type + Metadata_treatment
cfret_df = cfret_df.with_columns(
    (
        pl.col("Metadata_cell_type").cast(pl.Utf8)
        + "_"
        + pl.col("Metadata_treatment").cast(pl.Utf8)
    ).alias(treatment_col)
)

# split features
cfret_meta, cfret_feats = split_meta_and_features(cfret_df)

# Display data
print(f"Dataframe shape: {cfret_df.shape}")
cfret_df.head()

Dataframe shape: (15793, 679)


Metadata_cell_id,Metadata_WellRow,Metadata_WellCol,Metadata_heart_number,Metadata_cell_type,Metadata_heart_failure_type,Metadata_treatment,Metadata_Nuclei_Location_Center_X,Metadata_Nuclei_Location_Center_Y,Metadata_Cells_Location_Center_X,Metadata_Cells_Location_Center_Y,Metadata_Image_Count_Cells,Metadata_ImageNumber,Metadata_Plate,Metadata_Well,Metadata_Cells_Number_Object_Number,Metadata_Cytoplasm_Parent_Cells,Metadata_Cytoplasm_Parent_Nuclei,Metadata_Nuclei_Number_Object_Number,Metadata_Site,Cytoplasm_AreaShape_BoundingBoxMinimum_X,Cytoplasm_AreaShape_Compactness,Cytoplasm_AreaShape_Eccentricity,Cytoplasm_AreaShape_Extent,Cytoplasm_AreaShape_FormFactor,Cytoplasm_AreaShape_MajorAxisLength,Cytoplasm_AreaShape_MeanRadius,Cytoplasm_AreaShape_MinorAxisLength,Cytoplasm_AreaShape_Perimeter,Cytoplasm_AreaShape_Solidity,Cytoplasm_AreaShape_Zernike_0_0,Cytoplasm_AreaShape_Zernike_1_1,Cytoplasm_AreaShape_Zernike_2_0,Cytoplasm_AreaShape_Zernike_2_2,Cytoplasm_AreaShape_Zernike_3_1,Cytoplasm_AreaShape_Zernike_4_0,Cytoplasm_AreaShape_Zernike_4_2,…,Nuclei_Texture_DifferenceVariance_Mitochondria_3_03_256,Nuclei_Texture_DifferenceVariance_PM_3_03_256,Nuclei_Texture_InfoMeas1_ER_3_00_256,Nuclei_Texture_InfoMeas1_ER_3_01_256,Nuclei_Texture_InfoMeas1_ER_3_02_256,Nuclei_Texture_InfoMeas1_ER_3_03_256,Nuclei_Texture_InfoMeas1_Hoechst_3_00_256,Nuclei_Texture_InfoMeas1_Hoechst_3_01_256,Nuclei_Texture_InfoMeas1_Hoechst_3_02_256,Nuclei_Texture_InfoMeas1_Hoechst_3_03_256,Nuclei_Texture_InfoMeas1_Mitochondria_3_00_256,Nuclei_Texture_InfoMeas1_Mitochondria_3_01_256,Nuclei_Texture_InfoMeas1_Mitochondria_3_02_256,Nuclei_Texture_InfoMeas1_Mitochondria_3_03_256,Nuclei_Texture_InfoMeas1_PM_3_00_256,Nuclei_Texture_InfoMeas1_PM_3_01_256,Nuclei_Texture_InfoMeas1_PM_3_02_256,Nuclei_Texture_InfoMeas1_PM_3_03_256,Nuclei_Texture_InfoMeas2_ER_3_01_256,Nuclei_Texture_InfoMeas2_ER_3_03_256,Nuclei_Texture_InfoMeas2_Hoechst_3_01_256,Nuclei_Texture_InfoMeas2_Hoechst_3_03_256,Nuclei_Texture_InfoMeas2_PM_3_01_256,Nuclei_Texture_InfoMeas2_PM_3_03_256,Nuclei_Texture_InverseDifferenceMoment_Actin_3_02_256,Nuclei_Texture_InverseDifferenceMoment_ER_3_01_256,Nuclei_Texture_InverseDifferenceMoment_ER_3_03_256,Nuclei_Texture_InverseDifferenceMoment_Mitochondria_3_03_256,Nuclei_Texture_InverseDifferenceMoment_PM_3_01_256,Nuclei_Texture_InverseDifferenceMoment_PM_3_03_256,Nuclei_Texture_SumEntropy_PM_3_01_256,Nuclei_Texture_SumVariance_ER_3_03_256,Nuclei_Texture_SumVariance_Hoechst_3_03_256,Nuclei_Texture_SumVariance_Mitochondria_3_01_256,Nuclei_Texture_SumVariance_PM_3_01_256,Metadata_treatment_and_heart,Metadata_cell_type_and_treatment
str,str,i64,i64,str,str,str,f64,f64,f64,f64,i64,i64,str,str,i64,i64,i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
"""210e4376efde9d557a5c60029bdda6…","""B""",2,9,"""failing""","""rejected""","""DMSO""",221.046761,137.115493,246.6028,109.285755,40,1,"""localhost230405150001""","""B02""",1,1,6,6,"""f00""",-1.35494,0.841229,0.648883,-0.850138,-1.045214,1.298358,0.376165,0.935101,1.530228,-0.983617,-0.261031,-0.299817,-0.721977,0.944725,0.161074,0.532329,1.845864,…,0.797095,0.359081,-0.173336,0.300041,0.217945,-0.039774,0.488531,0.472164,0.28659,0.464359,0.501649,0.507623,1.076663,0.741941,-0.696022,-0.178762,0.186741,0.158222,0.341595,0.50487,-0.440604,-0.426966,0.194372,-0.035117,0.400021,-0.619206,-0.393448,0.961214,0.406068,0.374039,-0.280532,-0.158967,-0.344804,-0.263653,-0.305486,"""DMSO_heart_9""","""failing_DMSO"""
"""cef18f209640ef8ae98ec110cfdcb6…","""B""",2,9,"""failing""","""rejected""","""DMSO""",690.596142,183.067828,716.170091,177.132195,40,1,"""localhost230405150001""","""B02""",2,2,7,7,"""f00""",0.657107,-0.850399,-0.584931,2.090925,1.263259,-0.021031,1.627957,0.944161,-0.085511,1.475345,2.164761,-0.688462,1.215015,1.499086,-0.770667,1.012721,0.6791,…,-1.

## Split Populations

Separate cells into the reference (failing DMSO) and target (healthy DMSO) populations. The KS test will compare these two groups feature by feature.

In [6]:
# reference population: failing cardiomyocytes (DMSO-treated)
ref_df = cfret_df.filter(pl.col("Metadata_cell_type_and_treatment") == failing_label)

# target population: healthy cardiomyocytes (DMSO-treated)
target_df = cfret_df.filter(pl.col("Metadata_cell_type_and_treatment") == healthy_label)

## Run KS Test on Observed Data

Compute the KS statistic and FDR-corrected p-value for each morphology feature. Features with a corrected p-value below the significance threshold (alpha = 0.05) are labeled **"on"** and make up the morphology signature. All results are saved to a CSV file.

In [7]:
ks_results_df = compute_ks_signature(
    ref_df=ref_df,
    target_df=target_df,
    features=cfret_feats,
    output_path=signatures_results_dir / "signature_importance.csv",
)

print(ks_results_df.shape)
ks_results_df.head()

(657, 8)


feature,p_value,ks_stat,p_value_fdr_corrected,neg_log10_p_value,signature,compartment,channel
str,f64,f64,f64,f64,str,str,str
"""Cytoplasm_AreaShape_BoundingBo…",0.000128,0.063555,0.00015,3.825207,"""on""","""Cytoplasm""","""no-channel"""
"""Cytoplasm_AreaShape_Compactnes…",0.784666,0.018836,0.789473,0.102663,"""off""","""Cytoplasm""","""no-channel"""
"""Cytoplasm_AreaShape_Eccentrici…",3.8892e-47,0.211618,9.9039e-47,46.004195,"""on""","""Cytoplasm""","""no-channel"""
"""Cytoplasm_AreaShape_Extent""",1.8732e-21,0.142288,3.1395e-21,20.503135,"""on""","""Cytoplasm""","""no-channel"""
"""Cytoplasm_AreaShape_FormFactor""",0.784666,0.018836,0.789473,0.102663,"""off""","""Cytoplasm""","""no-channel"""


## Null Control: Shuffled Data

To verify that the observed signatures are not due to chance, we repeat the KS test on shuffled data. Shuffling randomly permutes feature values across all cells, breaking any real biological signal. We expect very few (or no) features to be labeled **"on"** in the shuffled results.

In [8]:
# combine reference and target into one dataframe before shuffling
concat_df = pl.concat([ref_df, target_df])

# shuffle feature values across all cells to remove biological signal (null control)
shuffled_concat_df = shuffle_feature_profiles(
    concat_df, cfret_feats, method="column", seed=0
)

# re-split the shuffled data into reference and target populations
shuffled_ref_df = shuffled_concat_df.filter(
    pl.col("Metadata_cell_type_and_treatment") == failing_label
)
shuffled_target_df = shuffled_concat_df.filter(
    pl.col("Metadata_cell_type_and_treatment") == healthy_label
)

In [9]:
ks_results_df = compute_ks_signature(
    ref_df=shuffled_ref_df,
    target_df=shuffled_target_df,
    features=cfret_feats,
    output_path=signatures_results_dir / "shuffle_signature_importance.csv",
)

print(ks_results_df.shape)
ks_results_df.head()

(657, 8)


feature,p_value,ks_stat,p_value_fdr_corrected,neg_log10_p_value,signature,compartment,channel
str,f64,f64,f64,f64,str,str,str
"""Cytoplasm_AreaShape_BoundingBo…",0.880392,0.016886,0.995973,0.001752,"""off""","""Cytoplasm""","""no-channel"""
"""Cytoplasm_AreaShape_Compactnes…",0.531722,0.023271,0.960051,0.017706,"""off""","""Cytoplasm""","""no-channel"""
"""Cytoplasm_AreaShape_Eccentrici…",0.432356,0.025141,0.948906,0.022777,"""off""","""Cytoplasm""","""no-channel"""
"""Cytoplasm_AreaShape_Extent""",0.539966,0.023124,0.960051,0.017706,"""off""","""Cytoplasm""","""no-channel"""
"""Cytoplasm_AreaShape_FormFactor""",0.628251,0.02158,0.976268,0.010431,"""off""","""Cytoplasm""","""no-channel"""
